# **Routing Pharmaceutical Document Queries Using Metadata (No Embeddings Needed)**
In this notebook, you'll learn how to route user queries to the most relevant PDF pages using simple metadata—without relying on embeddings. We'll upload a real multi-page pharmaceutical document, extract text from each page, classify the document type, and use metadata filtering to quickly match queries like “What are the part numbers listed?” to relevant content such as certificates of quality or packaging specifications. This approach keeps retrieval fast, explainable, and cost-effective.



In [1]:
# ============================================================
# STEP 1: Install Required Libraries
# ============================================================
#
# WHAT WE'RE DOING:
# Installing LlamaIndex and its PDF reader for loading
# pharmaceutical blob files.
#
# WHY THIS MATTERS:
# LlamaIndex provides the document loading and retrieval
# framework we'll use throughout this notebook.
#
# WHAT YOU'LL SEE:
# Package installation output (may take 1-2 minutes).
# ============================================================

!pip install -q llama-index
!pip install -q llama-index-readers-file

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 11.3 MB/s eta 0:00:00


Link to the sample pharmaceutical document used in this notebook.


In [2]:
# ============================================================
# STEP 2: Load the Pharmaceutical Blob PDF
# ============================================================
#
# WHAT WE'RE DOING:
# Loading the 10-page pharmaceutical blob PDF using
# LlamaIndex's PDFReader, which returns one Document per page.
#
# WHY THIS MATTERS:
# Each page becomes a separate document object, giving us
# page-level control for classification and routing.
#
# WHAT YOU'LL SEE:
# The total number of pages loaded and a preview of the
# first page's text content.
# ============================================================

from llama_index.readers.file import PDFReader
import time

# Load the PDF
loader = PDFReader()
documents = loader.load_data("/content/pharma-blob-sample.pdf")

# Preview to confirm it worked
print(f"Loaded {len(documents)} pages")
print(documents[0].text[:300])

Loaded 10 pages
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: AKTA ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and


**Step 1: Store File-Level Metadata**

When users upload pharmaceutical PDFs, don't just save the raw file—attach useful metadata that helps identify what the document contains.

In [3]:
# ============================================================
# STEP 3: Store File-Level Metadata
# ============================================================
#
# WHAT WE'RE DOING:
# Attaching metadata to each page: file ID, user ID, year,
# filename, page number, and a placeholder doc_type.
#
# WHY THIS MATTERS:
# This metadata is what enables filtering later — we can
# route queries by document type, year, or source file.
#
# WHAT YOU'LL SEE:
# The metadata dictionary for the first page, showing all
# the fields we've attached.
# ============================================================

import uuid

file_id = str(uuid.uuid4())  # Assigns Unique ID
user_id = "xyz"
year = "2024"
filename = "pharma-blob-sample.pdf"

pdf_metadata_store = []

for i, doc in enumerate(documents):
    metadata = {
        "file_id": file_id,
        "user_id": user_id,
        "doc_type": "unknown",  # We'll classify it later
        "year": year,
        "filename": filename,
        "page_number": i + 1,
        "text": doc.text
    }
    pdf_metadata_store.append(metadata)

print("Stored Metadata:")
print(pdf_metadata_store[0])

Stored Metadata:
{'file_id': 'e3ef4dcf-dbfa-4664-91a7-931109421c18', 'user_id': 'xyz', 'doc_type': 'unknown', 'year': '2024', 'filename': 'pharma-blob-sample.pdf', 'page_number': 1, 'text': 'Cytiva\n100 Results Way\nMarlborough, MA 01752\nUnited States\nPage 1 / 1\ncytiva.com\n3 June, 2022\nRe: AKTA ready Flow Kit Storage Conditions\nTo Whom It May Concern,\nThe recommended storage temperature for standard AKTA ready flow kits is provided in Section 8.3 of the Operating\nInstructions 28960345 and specified as > +5°C. This recommendation also applies to all modified AKTA ready flow kits\nas well, including the two listed in the below table. Extended storage below the recommended +5°C could lead to\nbrittleness or cracking of the plastic connectors. However, the operating temperature of AKTA ready flow kits is +2°C to\n+40°C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of\ndamage to the kit during setup and handling.\nDescript

**Step 2: Classify the User Query**

Before you search through documents, figure out what kind of pharmaceutical document the question is about. This helps you focus only on the most relevant files.

In [7]:
# ============================================================
# STEP 4: Set Up Gemini LLM Connection
# ============================================================
#
# WHAT WE'RE DOING:
# Creating a helper function that sends prompts to Google's
# Gemini 2.0 Flash model using your API key from Colab secrets.
#
# WHY THIS MATTERS:
# We'll use this function for both query classification and
# document type classification in the next steps.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================================

def gemini_model(prompt):
    import google.generativeai as genai
    from google.colab import userdata

    genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

    model = genai.GenerativeModel("models/gemini-2.5-flash")
    response = model.generate_content(prompt)

    return response.text

In [8]:
# ============================================================
# STEP 5: Classify User Query by Document Type
# ============================================================
#
# WHAT WE'RE DOING:
# Using the LLM to predict which pharmaceutical document type
# is most likely to answer the user's query.
#
# WHY THIS MATTERS:
# Instead of searching all pages, we narrow down to just the
# relevant document type first — faster and more accurate.
#
# WHAT YOU'LL SEE:
# The predicted document type (e.g., "packaging_specification").
# ============================================================

# Valid document type labels used throughout this notebook
VALID_DOC_TYPES = [
    "cover_letter", "certificate_of_quality", "packaging_specification",
    "bse_tse_declaration", "material_description", "supplier_qualification",
    "chain_of_custody", "unknown"
]

def clean_llm_label(response):
    """Clean up LLM response to extract a valid doc_type label."""
    cleaned = response.strip().replace('"', '').replace('`', '').replace('*', '').lower().replace(".", "").strip()
    # Check if the cleaned response contains a valid label
    for label in VALID_DOC_TYPES:
        if label in cleaned:
            return label
    return cleaned

def classify_query_llm(query, metadata_store):
    doc_list = "\n".join(
        [f"{i+1}. {doc['filename']} - doc_type: {doc['doc_type']}" for i, doc in enumerate(metadata_store)]
    )

    prompt = f"""
  You are an intelligent assistant that routes user queries to the most relevant pharmaceutical document.

  Available documents:
  {doc_list}

  User query: "{query}"

  Which document(s) are most likely to contain the answer?
  Respond with one of the following types:
  ["cover_letter", "certificate_of_quality", "packaging_specification", "bse_tse_declaration",
"material_description", "supplier_qualification", "chain_of_custody", "unknown"]

  Respond with ONLY the label. No explanation.
  """

    response = gemini_model(prompt)
    return clean_llm_label(response)

In [9]:
# ============================================================
# STEP 6: Test Query Classification
# ============================================================
#
# WHAT WE'RE DOING:
# Testing the classify_query_llm function with a sample
# pharmaceutical query about part numbers.
#
# WHY THIS MATTERS:
# We need to verify the LLM correctly identifies which
# document type should contain the answer.
#
# WHAT YOU'LL SEE:
# The predicted document type (e.g., "packaging_specification"
# or "certificate_of_quality").
# ============================================================

query = "What are the part numbers listed in this document?"
predicted_doc_type = classify_query_llm(query, pdf_metadata_store)
print(predicted_doc_type)

packaging_specification


Before moving on to Step 3, we first need to classify each PDF page based on its content and assign a doc_type using a simple rule-based function.



In [10]:
# ============================================================
# STEP 7: Classify Document Content
# ============================================================
#
# WHAT WE'RE DOING:
# Defining a function that classifies each page's content
# into one of the 7 pharmaceutical document types using the LLM.
#
# WHY THIS MATTERS:
# Once every page has a doc_type label, we can filter
# retrieval to only search within the right document type.
#
# WHAT YOU'LL SEE:
# No output (just defines the function).
# ============================================================

def classify_doc_type_llm(text, max_chars=1000):
    truncated_text = text[:max_chars]

    prompt = f"""
You are classifying a pharmaceutical document into one of the following types:
["cover_letter", "certificate_of_quality", "packaging_specification", "bse_tse_declaration",
"material_description", "supplier_qualification", "chain_of_custody", "unknown"]

Document content:
\"\"\"
{truncated_text}
\"\"\"

What is the best doc_type label for this document? Respond with ONLY one of the labels above. No explanation.
"""
    try:
        response = gemini_model(prompt)
        return clean_llm_label(response)
    except Exception as e:
        print("LLM failed:", e)
        return "unknown"

In [11]:
# ============================================================
# STEP 8: Apply Document Classification to All Pages
# ============================================================
#
# WHAT WE'RE DOING:
# Running the classifier on every page to assign a doc_type
# label (e.g., "certificate_of_quality", "bse_tse_declaration").
#
# WHY THIS MATTERS:
# This is the labeling step — once complete, each page in
# our metadata store has a meaningful document type tag.
#
# WHAT YOU'LL SEE:
# Progress messages as each page is classified (10 pages total).
# ============================================================

for i, doc in enumerate(pdf_metadata_store):
    print(f"Classifying doc {i+1}...")
    doc["doc_type"] = classify_doc_type_llm(doc["text"])
    time.sleep(1)  # Avoid rate limiting

Classifying doc 1...
Classifying doc 2...
Classifying doc 3...
Classifying doc 4...
Classifying doc 5...
Classifying doc 6...
Classifying doc 7...


LLM failed: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 52.188048386s.
Classifying doc 8...
Classifying doc 9...
Classifying doc 10...


**Step 3: Retrieve Files Matching That Doc Type**

Once you've classified the query, use metadata to fetch only the relevant pharmaceutical documents.

In [12]:
# ============================================================
# STEP 9: Retrieve Files by Document Type
# ============================================================
#
# WHAT WE'RE DOING:
# Filtering the metadata store to return only pages whose
# doc_type matches the predicted type from the user's query.
#
# WHY THIS MATTERS:
# This is the routing payoff — instead of searching all 10
# pages, we only look at pages of the right document type.
#
# WHAT YOU'LL SEE:
# The matched documents with their metadata and text content.
# ============================================================

def retrieve_files_by_doc_type(doc_type, metadata_store):
    return [doc for doc in metadata_store if doc["doc_type"] == doc_type]

# Debug: show what we're matching against
print(f"Query predicted type: '{predicted_doc_type}'")
print(f"Page types in store: {[doc['doc_type'] for doc in pdf_metadata_store]}")
print()

# Call the function using the predicted type
matched_files = retrieve_files_by_doc_type(predicted_doc_type, pdf_metadata_store)

# Display matched documents
print(f"Found {len(matched_files)} matching page(s):")
for doc in matched_files:
    print(f"  Page {doc['page_number']}: {doc['doc_type']} - {doc['text'][:100]}...")

Query predicted type: 'packaging_specification'
Page types in store: ['material_description', 'certificate_of_quality', 'certificate_of_quality', 'packaging_specification', 'packaging_specification', 'bse_tse_declaration', 'unknown', 'supplier_qualification', 'supplier_qualification', 'chain_of_custody']

Found 2 matching page(s):
  Page 4: packaging_specification - Packaging Component Specification
Document Number: PKG-SPEC-2023-0847
Revision: C
Effective Date: 20...
  Page 5: packaging_specification - Packaging Component Specification (continued)
Document Number: PKG-SPEC-2023-0847, Page 2 of 2
4. Co...


**Step 4 (Optional): Fall Back to Keyword-Based Search**

If your metadata filter still leaves too many results — or the user's query is vague — you can run a keyword-based search, but only within the smaller set of matching pharmaceutical documents.

In [14]:
from pprint import pprint
# ============================================================
# STEP 10: Fallback — Keyword-Based Search
# ============================================================
#
# WHAT WE'RE DOING:
# If metadata filtering returns too many results or the query
# is vague, we can narrow further with keyword matching.
#
# WHY THIS MATTERS:
# Keyword search is a simple fallback that works well for
# pharmaceutical terms like "part number", "lot", "BSE".
#
# WHAT YOU'LL SEE:
# The best matching document from the keyword search.
# ============================================================

def keyword_search(query, documents):
    query = query.lower()
    keywords = ["part number", "article number", "item", "specification",
                "product", "certificate", "lot", "description",
                "sterilization", "BSE", "supplier"]

    for doc in documents:
        text = doc["text"].lower()
        if any(keyword.lower() in text for keyword in keywords):
            return doc
    return None

user_query = "What part numbers are listed?"
print("Best matching result (fallback):")
pprint(keyword_search(user_query, matched_files))

Best matching result (fallback):
{'doc_type': 'packaging_specification',
 'file_id': 'e3ef4dcf-dbfa-4664-91a7-931109421c18',
 'filename': 'pharma-blob-sample.pdf',
 'page_number': 4,
 'text': 'Packaging Component Specification\n'
         'Document Number: PKG-SPEC-2023-0847\n'
         'Revision: C\n'
         'Effective Date: 2023-11-08\n'
         'Component: AKTA ready Flow Kit Packaging Assembly\n'
         'Drawing Reference: DWG-29477427-PKG-R3\n'
         '1. Scope\n'
         'This specification defines the packaging configuration for AKTA '
         'ready Flow Kit assemblies, including primary and\n'
         'secondary packaging components.\n'
         '2. Materials\n'
         'Component Material Specification\n'
         'Blister Tray PETG ASTM D1003, min 92% clarity\n'
         'Lid Film Tyvek 1073B Per DuPont specification\n'
         'Secondary Carton Corrugated ECT-32, C-flute\n'
         'Inner Cushion PE Foam 2 lb density, 1 inch thick\n'
         '3. Part Numbers\n